In [1]:
# !pip install deep-sort-realtime

In [ ]:
import os
from pathlib import Path
import csv
import cv2
import numpy as np
from ultralytics import YOLO
import unicodedata

# =========================
# CONFIG (edit here)
# =========================
PROJECT_ROOT = Path("raw_videos")
VIDEOS_RAW_DIR = PROJECT_ROOT / "videos_raw"
VIDEOS_OVERLAY_DIR = PROJECT_ROOT / "videos_overlay"
TRACKS_PIXEL_DIR = PROJECT_ROOT / "tracks_pixel"
LOGS_DIR = PROJECT_ROOT / "logs"

YOLO_WEIGHTS = "yolov8n.pt"   # try yolov8s.pt or v8m for stronger boxes
DEVICE = "cuda"               # "cuda", "cuda:0", or "cpu"
CONF_THRES = 0.25             # a bit higher to reduce junk/overlaps
IOU_THRES = 0.4              # NMS IoU

# Optional ROI polygon in PIXELS; set to None to disable
# Example:
# OBS_ROI_POLY = np.array([[120,100],[1180,100],[1180,700],[120,700]], dtype=np.int32)
OBS_ROI_POLY = None

DRAW_SCORES = True
SHOW_WINDOW = True
TARGET_FPS_OVERRIDE = None    # e.g., 30.0 to force output fps; else use input fps
# =========================


def ensure_dirs():
    VIDEOS_OVERLAY_DIR.mkdir(parents=True, exist_ok=True)
    TRACKS_PIXEL_DIR.mkdir(parents=True, exist_ok=True)
    LOGS_DIR.mkdir(parents=True, exist_ok=True)


def list_videos():
    exts = {".mp4", ".avi", ".mov", ".mkv"}
    vids = []
    if not VIDEOS_RAW_DIR.exists():
        return vids
    for p in VIDEOS_RAW_DIR.iterdir():
        if p.is_file() and p.suffix.lower() in exts:
            vids.append(p)
    return sorted(vids)


def centroid_in_roi(x1, y1, x2, y2, roi_poly):
    if roi_poly is None:
        return True
    cx, cy = (x1 + x2) * 0.5, (y1 + y2) * 0.5
    return cv2.pointPolygonTest(roi_poly, (float(cx), float(cy)), False) >= 0


def is_finite_box(x1, y1, x2, y2):
    vals = np.array([x1, y1, x2, y2], dtype=float)
    if not np.all(np.isfinite(vals)):
        return False
    if x2 <= x1 or y2 <= y1:
        return False
    return True


def clamp_box(x1, y1, x2, y2, W, H):
    x1 = float(min(max(x1, 0.0), W))
    y1 = float(min(max(y1, 0.0), H))
    x2 = float(min(max(x2, 0.0), W))
    y2 = float(min(max(y2, 0.0), H))
    return x1, y1, x2, y2


def to_ascii_safe(s):
    if s is None:
        return "unknown"
    if not isinstance(s, str):
        s = str(s)
    # strip control chars to avoid OSError on some Windows setups
    s = "".join(ch for ch in s if unicodedata.category(ch)[0] != "C")
    return s


def iou(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    a_area = max(0.0, (ax2 - ax1)) * max(0.0, (ay2 - ay1))
    b_area = max(0.0, (bx2 - bx1)) * max(0.0, (by2 - by1))
    union = a_area + b_area - inter + 1e-9
    return inter / union


def non_max_suppression_per_id(xyxy, tids, confs, iou_thr=0.9):
    """
    Safety net: if ByteTrack somehow yields two boxes for the same track in one frame,
    keep the higher-confidence one when IoU is very high.
    Returns indices to keep.
    """
    idxs = list(range(len(tids)))
    keep = []
    used = set()
    for i in idxs:
        if i in used:
            continue
        group = [j for j in idxs if tids[j] == tids[i]]
        group_sorted = sorted(group, key=lambda j: confs[j], reverse=True)
        while group_sorted:
            top = group_sorted.pop(0)
            keep.append(top)
            used.add(top)
            rest = []
            for j in group_sorted:
                if iou(xyxy[top], xyxy[j]) <= iou_thr:
                    rest.append(j)
                else:
                    used.add(j)
            group_sorted = rest
    return keep


def process_video_with_bytetrack(video_path: Path, model: YOLO, names: dict):
    video_id = video_path.stem
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Failed to open {video_path}")

    in_fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    fps = TARGET_FPS_OVERRIDE or in_fps
    W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    out_overlay = VIDEOS_OVERLAY_DIR / f"{video_id}_overlay.mp4"
    out_csv = TRACKS_PIXEL_DIR / f"{video_id}.csv"

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(str(out_overlay), fourcc, fps, (W, H))

    # CSV header
    with open(out_csv, "w", newline="", encoding="utf-8") as fcsv:
        w = csv.writer(fcsv)
        w.writerow(["video_id", "frame_idx", "t", "track_id", "class", "conf_det", "u", "v", "w", "h"])

        frame_idx = 0
        try:
            while True:
                ok, frame = cap.read()
                if not ok:
                    break
                t_sec = frame_idx / in_fps

                # ---- Ultralytics tracker (ByteTrack) ----
                res_list = model.track(
                    source=frame,
                    persist=True,
                    device=DEVICE,
                    conf=CONF_THRES,
                    iou=IOU_THRES,
                    tracker="bytetrack.yaml",
                    verbose=False
                )

                vis = frame.copy()

                if res_list and res_list[0].boxes is not None and res_list[0].boxes.id is not None:
                    boxes = res_list[0].boxes

                    xyxy = boxes.xyxy.cpu().numpy()
                    tids = boxes.id.int().cpu().numpy().tolist()
                    clss = boxes.cls.cpu().numpy().astype(int) if boxes.cls is not None else np.zeros(len(tids), dtype=int)
                    confs = boxes.conf.cpu().numpy() if boxes.conf is not None else np.ones(len(tids), dtype=float)

                    # Safety de-dup per track id (rare with ByteTrack, but keep it)
                    keep = non_max_suppression_per_id(xyxy, tids, confs, iou_thr=0.90)

                    for i in keep:
                        x1, y1, x2, y2 = map(float, xyxy[i])
                        tid = int(tids[i])
                        cid = int(clss[i])
                        c = float(confs[i])

                        # Clamp to image and validate box
                        x1, y1, x2, y2 = clamp_box(x1, y1, x2, y2, W, H)
                        if not is_finite_box(x1, y1, x2, y2):
                            continue

                        if not centroid_in_roi(x1, y1, x2, y2, OBS_ROI_POLY):
                            continue

                        bw, bh = (x2 - x1), (y2 - y1)
                        # Resolve class name robustly
                        cls_name = names.get(cid, str(cid)) if isinstance(names, dict) else str(cid)
                        cls_name = to_ascii_safe(cls_name)

                        # ---- CSV row (unique track per frame) ----
                        try:
                            w.writerow([
                                video_id, frame_idx, f"{t_sec:.6f}", tid, cls_name, f"{c:.4f}",
                                f"{x1:.1f}", f"{y1:.1f}", f"{bw:.1f}", f"{bh:.1f}"
                            ])
                        except OSError:
                            # Very defensive: strip to ASCII-only & try again
                            w.writerow([
                                to_ascii_safe(video_id), frame_idx, f"{t_sec:.6f}", tid, to_ascii_safe(cls_name), f"{c:.4f}",
                                f"{x1:.1f}", f"{y1:.1f}", f"{bw:.1f}", f"{bh:.1f}"
                            ])

                        # ---- draw overlay ----
                        color = (0, 255, 0) if cls_name == "person" else (255, 128, 0)
                        cv2.rectangle(vis, (int(x1), int(y1)), (int(x2), int(y2)), color, 2)
                        label = f"id:{tid} {cls_name}"
                        if DRAW_SCORES:
                            label += f" {c:.2f}"
                        cv2.putText(vis, label, (int(x1), max(0, int(y1) - 6)),
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2, cv2.LINE_AA)

                # (Optional) draw ROI border
                if OBS_ROI_POLY is not None:
                    cv2.polylines(vis, [OBS_ROI_POLY], isClosed=True, color=(0, 255, 255), thickness=2)

                writer.write(vis)
                if SHOW_WINDOW:
                    cv2.imshow("overlay", vis)
                    if cv2.waitKey(1) == 27:
                        break

                frame_idx += 1

        finally:
            cap.release()
            writer.release()
            if SHOW_WINDOW:
                cv2.destroyAllWindows()

    print(f"[DONE] {video_path.name} -> CSV: {out_csv} | Overlay: {out_overlay} | FPS: {fps:.2f} | Frames: {total_frames}")


def main():
    ensure_dirs()
    model = YOLO(YOLO_WEIGHTS).to(DEVICE)
    names = model.names if hasattr(model, "names") else {}

    vids = list_videos()
    if not vids:
        print(f"No videos found in {VIDEOS_RAW_DIR.resolve()}")
        return

    for vp in vids:
        process_video_with_bytetrack(vp, model, names)


if __name__ == "__main__":
    main()
